# 14 Training results and comparison
Collects `runs/*/result.json`. Screening decisions use **validation** columns only; test columns are for locked, confirmed configurations.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from moebase import paths

import pandas as pd
rows = []
for res in sorted(paths.RUNS_DIR.glob("*/result.json")):
    r = json.loads(res.read_text()); rec = json.loads((res.parent / "record.json").read_text())
    row = {"run": res.parent.name, "config": rec["name"], "seed": rec["seed"], "params": rec["n_params"],
           "best_step": r["best_step"], "valid_brier": r["best_valid_brier"], "valid_acc": r["valid"]["acc"]}
    for part, m in r.get("test", {}).items():
        row[f"{part}_acc"] = m["acc"]
        if "acc_excl_swap_dup" in m:
            row[f"{part}_acc_nodup"] = m["acc_excl_swap_dup"]
    rows.append(row)
df = pd.DataFrame(rows); df

In [ ]:
# seed-level and mean validation results per config (screening view)
if len(df):
    display(df.groupby("config")[["valid_brier", "valid_acc"]].agg(["mean", "std", "count"]))

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4))
for res in sorted(paths.RUNS_DIR.glob("*/result.json")):
    r = json.loads(res.read_text())
    if "test_len" not in r.get("test", {}):
        continue
    pl = {**r["test"]["test_iid"]["per_length"], **r["test"]["test_len"]["per_length"]}
    xs = sorted(int(k) for k in pl)
    ax.plot(xs, [pl[str(x)]["acc"] if str(x) in pl else pl[x]["acc"] for x in xs], marker=".", label=res.parent.name[:30])
ax.axvline(6.5, color="grey", ls="--"); ax.set_xlabel("operators"); ax.set_ylabel("accuracy"); ax.legend(fontsize=7)